### 🎯 Module Overview
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

Table of Contents

- Introduction to Data Ingestion
- Text Files (.txt)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

---

### 💡 Interview & Learning Notes

**Key Interview Questions:**
1. *What is the difference between CharacterTextSplitter and RecursiveCharacterTextSplitter?* 
   - Character splits rigidly by a single character (like `\n`). Recursive tries a list of characters (e.g., `\n\n`, `\n`, ` `) iteratively until the chunk is small enough, preserving logical structure (paragraphs, sentences, words).
2. *Why do we need chunk overlap?* 
   - To maintain context across boundaries. If a key concept spans across a split, the overlap ensures at least one chunk captures the context.
3. *What is the significance of Metadata in a Document?* 
   - Metadata enables pre-filtering before vector similarity search (hybrid search), tracks data lineage, and provides necessary context to the LLM (e.g., date, author, page number).

**Learning Takeaways:**
- **Data Ingestion** is the foundation of RAG. Garbage in, garbage out.
- **TokenTextSplitter** is strictly aligned with the LLM's context window, whereas **RecursiveCharacterTextSplitter** preserves human readability. Use Recursive as your default.

### Introduction To Data Ingestion

In [ ]:
import os
from typing import List, Dict, Any
import pandas as pd

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

# Component Explanations:
# 1. Document (from langchain_core.documents):
#    - Purpose: LangChain's base class for storing text and its associated metadata.
#    - Why Used: Standardizes data passing between loaders, splitters, embedders, and vector stores.
# 2. RecursiveCharacterTextSplitter (from langchain_text_splitters):
#    - Purpose: Splits text using a hierarchy of separators (e.g., double newline, newline, space).
#    - Current Best Practice: This is the recommended default text splitter for general text because it tries to keep paragraphs and sentences together.
# 3. TokenTextSplitter:
#    - Purpose: Splits text based on token count (using libraries like tiktoken).
#    - Why Used: Ensures chunks fit precisely within an LLM's context window limit.

print("Setup completed!")

Setup completed!


### Understanding Document Structure In Langchain

In [ ]:
# Create document
doc = Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Krish Naik",
        "date_created": "2024-01-01",
        "custom_field": "any_value"
    }
)

print("\nDocument Structure")
print("-" * 40)

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

# Metadata importance
print("\nMetadata is crucial for:")
print("• Filtering search results")
print("• Tracking document sources")
print("• Providing context in responses")
print("• Debugging and auditing")

# Example splitter usage
splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=5
)

chunks = splitter.split_documents([doc])

print("\nSplit Chunks:")
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk.page_content)


Document Structure
----------------------------------------
Content: This is the main text content that will be embedded and searched.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01', 'custom_field': 'any_value'}

Metadata is crucial for:
• Filtering search results
• Tracking document sources
• Providing context in responses
• Debugging and auditing

Split Chunks:

Chunk 1:
This is the main text content

Chunk 2:
that will be embedded and

Chunk 3:
and searched.


In [ ]:
type(doc)

langchain_core.documents.base.Document

### Text Files (.txt) - The Simplest Case {#2-text-files}

In [ ]:
## Create a simple txt file
import os
os.makedirs("data/text_files",exist_ok=True)

In [ ]:
sample_texts={
    "data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


### TextLoader- Read Single File 

In [ ]:
from langchain_community.document_loaders import TextLoader

# Component Explanation: TextLoader
# - Purpose: Loads plain text files (.txt, .md, etc.).
# - Internally uses standard Python file I/O operations.
# - Converts raw text into LangChain Document objects with basic metadata (source path).
#

## Loading a single text file
loader=TextLoader(
    "data/text_files/python_intro.txt",
    encoding="utf-8"
)

documents=loader.load()
print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

📄 Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/text_files/python_intro.txt'}


C:\Users\DELL\AppData\Local\Temp\ipykernel_6340\271831367.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### DirectoryLoader- Multiple Text Files

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

# Component Explanation: DirectoryLoader
# - Purpose: Recursively loads all files in a directory that match a specific glob pattern.
# - Internally: Iterates over files and applies a specified loader (e.g., TextLoader) to each.
# - Why Used: Essential for batch ingestion of local knowledge bases.

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True

)

documents=dir_loader.load()

print(f"📁 Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata.get('source','Unknown')}")
    print(f"  Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 190.76it/s]

📁 Loaded 2 documents

Document 1:
  Source: data\text_files\machine_learning.txt
  Length: 575 characters

Document 2:
  Source: data\text_files\python_intro.txt
  Length: 489 characters

📊 DirectoryLoader Characteristics:
✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


### Text Splitting Statergies

In [ ]:
### Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprog

In [ ]:
### MEthod 1- Character Text Splitter
text=documents[0].page_content
text

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '

In [ ]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")

char_splitter = CharacterTextSplitter(
    separator=" ",  # Split on spaces
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [ ]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
------------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:


In [ ]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")

char_splitter = CharacterTextSplitter(
    separator="\n",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 4 chunks
First chunk: Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...


In [ ]:
print(char_chunks[0])
print("-------------")
print(char_chunks[1])
print("-------------")
print(char_chunks[2])

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
-------------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties


In [ ]:
# Method 2: Recursive character splitting (RECOMMENDED)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""],  # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks=recursive_splitter.split_text(text)

print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
Created 6 chunks
First chunk: Machine Learning Basics...


In [ ]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])

Machine Learning Basics
-----------------
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
------------------
that can access data and use it to learn for themselves.


In [ ]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[". "," ",""],  # Try sentence, word, then character splitting
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks=splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks)-1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")

    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long'
Chunk 2: '. This is sentence two and it is also quite long'

Chunk 2: '. This is sentence two and it is also quite long'
Chunk 3: '. This is sentence three which is even longer than the others'

Chunk 3: '. This is sentence three which is even longer than the others'
Chunk 4: '. This is sentence four. This is sentence five. This is sentence six.'



In [ ]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks=token_splitter.split_text(text)

print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [ ]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")

print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ Consistent chunking for LLMs")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ Consistent chunking for LLMs
  ❌ Slower than character-based
  Use when: Working with token-limited models


In [ ]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\DELL\Desktop\rag_praacties
['# %% [markdown].py', '.git', '.python-version', '.venv', '1-dataingestion.ipynb', '2-dataparsingpdf.ipynb', '3-dataparsingdoc.ipynb', '4-csvexcelparsing.ipynb', '5-jsonparsing.ipynb', '6-databaseparsing.ipynb', 'data', 'main.py', 'pyproject.toml', 'README.md', 'requirements.txt']


### 🚀 Best Practices for Data Ingestion

1. **Cost Efficiency & Token Optimization**:
   - Filter out noise (e.g., headers, footers, boilerplate) *before* ingestion.
   - Use `TokenTextSplitter` primarily as a fallback or strict token-boundary enforcer to avoid paying for padded or truncated tokens at the embedding stage.

2. **Time Optimization**:
   - Use batch processing (like `DirectoryLoader`) when dealing with large directories, but consider asynchronous loaders for high I/O wait times.
   - Monitor memory usage: `DirectoryLoader` loads all documents into RAM by default. For massive datasets, use lazy loading `.lazy_load()` if supported by the loader to yield documents iteratively.

3. **Metadata Enrichment**:
   - Always inject rich metadata (e.g., document type, creation date, chunk index) at ingestion time. It's much harder to add this later in the pipeline.
   - Clean your filenames and paths, as they are often automatically captured as the 'source' metadata.